# Notebook 03: Pemodelan, Hyperparameter Tuning, Evaluasi Forensik & Ekspor `.joblib` (Bab 4 & Bab 5 Laporan AM552)
**Mata Kuliah**: AM552 - Data Science of IS (Program Studi S1 Sistem Informasi, FTI UKSW)  
**Kode Kelompok**: `682024046_KEL_DS`

---
## 4.1 Pilihan Model (1 Baseline Sederhana + 2 Model Alternatif) & 4.2 Hyperparameter Tuning 5-Fold
1. **Model 0 (Dummy Baseline)**: `DummyClassifier(strategy='most_frequent')` sebagai batas bawah statistik.
2. **Model 1 (Baseline Sederhana — Bab 4.1)**: `LogisticRegression(class_weight='balanced')` dengan ekstraksi fitur `TfidfVectorizer(ngram_range=(1, 2))`.
3. **Model 2 (Alternatif 1 — Probabilistic NLP)**: `ComplementNB` (varian Multinomial Naive Bayes yang dioptimalkan untuk distribusi kelas teks yang timpang).
4. **Model 3 (Alternatif 2 — Margin Classifier)**: `CalibratedClassifierCV(LinearSVC(class_weight='balanced'))`.

In [1]:
import joblib, json
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Muat ringkasan hasil training & tuning 5-fold serta artefak model .joblib
with open('../models/modeling_results_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
artifact = joblib.load('../models/genai_absa_pipeline.joblib')
print('Model Terbaik Tersimpan:', artifact['model_name'])
print('Best Hyperparameters (5-Fold CV):', summary['best_params'])
pd.DataFrame(summary['model_comparison'])

Model Terbaik Tersimpan: Tuned Logistic Regression (TF-IDF 1-2 Gram, C=1.0, Balanced)
Best Hyperparameters (5-Fold CV): {'clf__C': 1.0}

=== TABEL PERBANDINGAN HASIL MODEL (BAB 5.2) ===
- 0. DummyClassifier (Most Frequent Baseline) | Train Macro-F1: 0.2776 | Test Macro-F1: 0.2776 | Test Weighted-F1: 0.5941 | Test ROC-AUC: 0.5000 | MCC: 0.0000
- 1. Logistic Regression (Tuned C=1.0)       | Train Macro-F1: 0.7617 | Test Macro-F1: 0.6512 | Test Weighted-F1: 0.8676 | Test ROC-AUC: 0.8751 | MCC: 0.6822
- 2. Complement Naive Bayes (Tuned alpha=0.2) | Train Macro-F1: 0.6357 | Test Macro-F1: 0.5818 | Test Weighted-F1: 0.8476 | Test ROC-AUC: 0.8713 | MCC: 0.6901
- 3. Calibrated LinearSVC (C=0.5, Balanced Margin) | Train Macro-F1: 0.6272 | Test Macro-F1: 0.5999 | Test Weighted-F1: 0.8847 | Test ROC-AUC: 0.8827 | MCC: 0.7623

## 5.3 Audit Forensik Metrik (`metric_sanity_audit.py`), Analisis Kesalahan & Fitur Terpenting
Menjalankan pemeriksaan 6 gerbang anti-leakage untuk membuktikan tidak ada kebocoran target (`score` atau `app_name`) dan mengevaluasi performa pada ulasan unik dan ulasan substantif ($\ge 5$ kata).

In [2]:
import sys
sys.path.insert(0, '../05_evaluation')
import metric_sanity_audit

with open('../05_evaluation/audit_results.json', 'r', encoding='utf-8') as f:
    audit_res = json.load(f)
print('Status 6 Gerbang Forensik Anti-Leakage:')
for k in ['audit_1_leakage', 'audit_2_imbalance_dummy', 'audit_3_duplicate_and_length_slices', 'audit_4_leave_one_app_out', 'audit_5_split_hygiene', 'audit_6_permutation_sanity']:
    print(f"  - {audit_res[k]['audit_name']}: PASSED={audit_res[k]['passed']}")

Status 6 Gerbang Forensik Anti-Leakage:
  - AUDIT-1: Circular / Target Leakage Check: PASSED=True
  - AUDIT-2: Class Imbalance Illusion vs. Dummy Baseline: PASSED=True
  - AUDIT-3: Exact-Text Duplicate & Substantive Review (>=5 Words) Slice Audit: PASSED=True
  - AUDIT-4: Leave-One-App-Out (LOAO) Cross-Domain Generalization: PASSED=True
  - AUDIT-5: Pipeline Split Hygiene & Zero-ID Overlap Check: PASSED=True
  - AUDIT-6A: Randomized-Label Permutation Sanity Test (y_shuffled): PASSED=True